# Notebook 3: Building a Knowledge Graph from PDFs

In this notebook, you'll learn how to:
1. Extract text from PDFs using **PyMuPDF** (with automatic OCR fallback for scanned documents)
2. Build a knowledge graph from the extracted text
3. Process multiple PDFs into a single graph

## How PDF Processing Works

The pipeline follows two stages:

```
PDF file → PyMuPDF text extraction (with OCR fallback) → Plain text
Plain text → Split into chunks → LLM extracts entities → Write to Neo4j
```

**Text extraction uses a 3-tier strategy per page:**
1. **PyMuPDF `get_text()`** — fast, works for text-based PDFs
2. **Tesseract OCR** — handles scanned/image-based pages
3. **Docling** — last-resort fallback for complex layouts and formats

## Step 1: Setup

In [8]:
import os
import asyncio
from pathlib import Path
import neo4j
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

driver = neo4j.GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)
driver.verify_connectivity()
print("Connected to Neo4j!")

Connected to Neo4j!


In [9]:
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

llm = OpenAILLM(
    model_name="gpt-4.1-2025-04-14",
    model_params={
        "temperature": 0,
        "response_format": {"type": "json_object"},
    },
)

embedder = OpenAIEmbeddings(model="text-embedding-3-large")
print("LLM and Embedder ready!")

LLM and Embedder ready!


## Step 2: PDF Text Extraction Setup

We use [PyMuPDF](https://pymupdf.readthedocs.io/en/latest/) for text extraction, with automatic fallback to OCR for scanned documents.

**Prerequisites for OCR (optional — only needed for scanned PDFs):**
- **Tesseract**: `brew install tesseract` (macOS) or `apt install tesseract-ocr` (Linux)
- **Docling**: `pip install docling` (last-resort fallback for complex layouts)

In [10]:
import pymupdf
import shutil
import subprocess
import tempfile
from pathlib import Path

data_dir = Path("../data")

# Verify Tesseract is available (optional — only needed for scanned PDFs)
if shutil.which("tesseract"):
    print(f"Tesseract found: {shutil.which('tesseract')}")
else:
    print("Tesseract not found (OCR fallback will be skipped).")
    print("  Install: brew install tesseract  (macOS) or apt install tesseract-ocr (Linux)")

# Verify Docling is available (optional — last-resort fallback)
try:
    from docling.document_converter import DocumentConverter
    print("Docling found")
except ImportError:
    print("Docling not installed (third fallback will be skipped).")
    print("  Install: pip install docling")

# Check available PDFs
pdf_files = sorted(data_dir.glob("*.pdf"))
print(f"\nFound {len(pdf_files)} PDF(s) in {data_dir.resolve()}:")
for pdf_file in pdf_files:
    doc = pymupdf.open(pdf_file)
    print(f"  {pdf_file.name}: {len(doc)} pages")
    doc.close()

Tesseract found: /opt/homebrew/bin/tesseract
Docling not installed (third fallback will be skipped).
  Install: pip install docling

Found 251 PDF(s) in /Users/henryadamcollie/Documents/GitHub/KG-pipe/data:
  Patent - 2010 - Dynamic indexing.pdf: 37 pages
  Patent - 2011 - Creating data in a data store using a dynamic ontology.pdf: 16 pages
  Patent - 2012 - Filter chains with associated views for exploring large data sets.pdf: 30 pages
  Patent - 2013 - Creating data in a data store using a dynamic ontology.pdf: 17 pages
  Patent - 2014 - Context-sensitive views.pdf: 26 pages
  Patent - 2014 - Data processing techniques 2.pdf: 40 pages
  Patent - 2014 - Data processing techniques. 1.pdf: 63 pages
  Patent - 2014 - Event matrix based on integrated data.pdf: 20 pages
  Patent - 2014 - Filter chains with associated multipath views for exploring large data sets.pdf: 84 pages
  Patent - 2014 - IP reputation.pdf: 23 pages
  Patent - 2014 - Improved data integration tool.pdf: 27 pages
  Pate

### Extraction Functions

These three functions implement the 3-tier extraction strategy. Each page is tried with PyMuPDF first, then OCR, then Docling as a last resort.

In [11]:
def extract_page_text(page):
    """Extract text from a single page, falling back to OCR if needed.

    1. get_text() — fast, works for native text PDFs.
    2. get_textpage_ocr() — PyMuPDF built-in Tesseract OCR.
    3. pixmap → PNG → tesseract subprocess — handles CCITTFax/JBIG2 1-bit fax images.

    Returns (text, method) where method is "text", "ocr", or "empty".
    """
    # Tier 1: Native text extraction
    text = page.get_text()
    if text.strip():
        return text, "text"

    # Tier 2: PyMuPDF built-in OCR
    try:
        tp = page.get_textpage_ocr(tessdata=None, language="eng", dpi=300)
        ocr_text = page.get_text(textpage=tp)
        if ocr_text.strip():
            return ocr_text, "ocr"
    except Exception:
        pass

    # Tier 3: Render page → PNG → Tesseract subprocess
    try:
        pix = page.get_pixmap(dpi=300)
        with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as f:
            tmpname = f.name
        pix.save(tmpname)
        result = subprocess.run(
            ["tesseract", tmpname, "stdout", "--psm", "3", "-l", "eng"],
            capture_output=True, text=True, timeout=60,
        )
        os.unlink(tmpname)
        if result.returncode == 0 and result.stdout.strip():
            return result.stdout, "ocr"
    except Exception:
        pass

    return "", "empty"


def extract_with_docling(pdf_path):
    """Last-resort extraction using Docling for the entire document."""
    try:
        from docling.document_converter import DocumentConverter
        converter = DocumentConverter()
        result = converter.convert(str(pdf_path))
        text = result.document.export_to_markdown()
        if text.strip():
            return text
    except Exception as e:
        print(f"  Docling failed for {pdf_path.name}: {e}")
    return None


def extract_text_from_pdf(pdf_path):
    """Extract text from a PDF using a 3-tier fallback strategy.

    Tier 1: PyMuPDF get_text() per page (fast, native text)
    Tier 2: Tesseract OCR per page (scanned/image pages)
    Tier 3: Docling on the full document (complex layouts, last resort)

    Returns (full_text, stats) where stats has page counts by method.
    """
    doc = pymupdf.open(pdf_path)
    pages = []
    stats = {"text": 0, "ocr": 0, "docling": 0, "empty": 0, "total": len(doc)}
    empty_indices = []

    for i, page in enumerate(doc):
        page_text, method = extract_page_text(page)
        pages.append(page_text)
        if method == "empty":
            empty_indices.append(i)
        stats[method] += 1

    doc.close()

    # Tier 3: If any pages are still empty, try Docling on the full document
    if empty_indices:
        docling_text = extract_with_docling(pdf_path)
        if docling_text:
            pages[empty_indices[0]] = docling_text
            stats["docling"] += 1
            stats["empty"] -= 1
            for idx in empty_indices[1:]:
                pages[idx] = ""
                stats["docling"] += 1
                stats["empty"] -= 1

    return "\n\n".join(pages), stats

print("Extraction functions ready.")

Extraction functions ready.


## Step 3: Define Schema & Build the Knowledge Graph

You can choose to define a manual schema or let the LLM figure one out automatically. Then we extract text from each PDF with PyMuPDF and feed it to the pipeline.

In [12]:
# ── Toggle: set to False to let the LLM generate its own schema ──
USE_MANUAL_SCHEMA = False

NODE_TYPES = [
    "Person",
    {
        "label": "Organization",
        "description": "A company, university, or research lab",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Model",
        "description": "A machine learning model or architecture",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Technique",
        "description": "A method, algorithm, or technique",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Dataset",
        "description": "A dataset or benchmark used for evaluation",
        "properties": [{"name": "name", "type": "STRING"}],
    },
    {
        "label": "Metric",
        "description": "An evaluation metric (e.g., BLEU, perplexity)",
        "properties": [{"name": "name", "type": "STRING"}],
    },
]

RELATIONSHIP_TYPES = [
    "AUTHORED_BY",
    "AFFILIATED_WITH",
    "USES_TECHNIQUE",
    "EVALUATED_ON",
    "ACHIEVES_SCORE_ON",
    "COMPARED_TO",
    {"label": "IMPROVES_UPON", "description": "One model/technique outperforms another"},
]

PATTERNS = [
    ("Model", "AUTHORED_BY", "Person"),
    ("Person", "AFFILIATED_WITH", "Organization"),
    ("Model", "USES_TECHNIQUE", "Technique"),
    ("Model", "EVALUATED_ON", "Dataset"),
    ("Model", "ACHIEVES_SCORE_ON", "Metric"),
    ("Model", "COMPARED_TO", "Model"),
    ("Model", "IMPROVES_UPON", "Model"),
]

if USE_MANUAL_SCHEMA:
    schema = {
        "node_types": NODE_TYPES,
        "relationship_types": RELATIONSHIP_TYPES,
        "patterns": PATTERNS,
    }
    print(f"Using MANUAL schema: {len(NODE_TYPES)} node types, {len(RELATIONSHIP_TYPES)} relationship types")
else:
    schema = "EXTRACTED"
    print("Using AUTO schema: the LLM will extract a schema from the PDF")

Using AUTO schema: the LLM will extract a schema from the PDF


### Extraction Error Handling

The LLM sometimes returns responses that don't match the expected JSON schema. We add:
- **Logging** — prints the raw LLM response when extraction fails, so you can see what went wrong
- **Retries** — up to 3 attempts per chunk before giving up

In [ ]:
import json
import logging
import sys
from functools import wraps
from neo4j_graphrag.experimental.components.entity_relation_extractor import (
    LLMEntityRelationExtractor,
    OnError,
)
from neo4j_graphrag.experimental.components.types import Neo4jGraph

MAX_RETRIES = 3

# Route the extractor's debug/error logs to notebook output
extractor_logger = logging.getLogger(
    "neo4j_graphrag.experimental.components.entity_relation_extractor"
)
extractor_logger.setLevel(logging.DEBUG)
if not any(isinstance(h, logging.StreamHandler) for h in extractor_logger.handlers):
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.DEBUG)
    handler.setFormatter(logging.Formatter("  [%(levelname)s] %(message)s"))
    extractor_logger.addHandler(handler)


# ── Fix: strip None property values from LLM responses ──
# The LLM returns None for properties it can't extract (e.g. 'start_date': None),
# but the Neo4jGraph Pydantic model doesn't accept None as a PropertyValue.
# This wrapper strips nulls before the package validates the response.
original_ainvoke = llm.ainvoke

async def ainvoke_strip_nulls(*args, **kwargs):
    result = await original_ainvoke(*args, **kwargs)
    try:
        data = json.loads(result.content)
        for node in data.get("nodes", []):
            if "properties" in node:
                node["properties"] = {
                    k: v for k, v in node["properties"].items() if v is not None
                }
        for rel in data.get("relationships", []):
            if "properties" in rel:
                rel["properties"] = {
                    k: v for k, v in rel["properties"].items() if v is not None
                }
        result.content = json.dumps(data)
    except (json.JSONDecodeError, KeyError, TypeError):
        pass  # Let the package handle malformed responses
    return result

llm.ainvoke = ainvoke_strip_nulls


def add_retries(extractor, max_retries=MAX_RETRIES):
    """Patch an extractor's extract_for_chunk to retry on failure with logging."""
    original_method = extractor.extract_for_chunk
    extractor.on_error = OnError.IGNORE

    @wraps(original_method)
    async def extract_with_retry(schema, examples, chunk):
        for attempt in range(max_retries):
            result = await original_method(schema, examples, chunk)
            if result.nodes or result.relationships:
                return result
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} for chunk_index={chunk.index}")
                print(f"    Chunk text preview: {chunk.text[:200]}...")
            else:
                print(f"  All {max_retries} attempts failed for chunk_index={chunk.index}")
        return result

    extractor.extract_for_chunk = extract_with_retry
    return extractor

print(f"Error handling ready (null stripping + max {MAX_RETRIES} retries per chunk).")

In [ ]:
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

kg_builder_pdf = SimpleKGPipeline(
    llm=llm,
    driver=driver,
    embedder=embedder,
    schema=schema,
    from_pdf=False,  # We extract text ourselves with PyMuPDF
    on_error="IGNORE",  # Let failures log details, our wrapper handles retries
    perform_entity_resolution=False,  # We'll run resolution manually in Step 4
    neo4j_database=NEO4J_DATABASE,
)

# Patch the extractor with retry logic
extractor = kg_builder_pdf.runner.pipeline._nodes["extractor"].component
add_retries(extractor)
print("Pipeline created with retry-enabled extractor.")

# Process all PDFs in the data folder
for pdf_path in sorted(data_dir.glob("*.pdf")):
    print(f"\nProcessing: {pdf_path.name}")

    # Step 1: Extract text with PyMuPDF (OCR fallback if needed)
    text, stats = extract_text_from_pdf(pdf_path)
    print(f"  Extracted {stats['total']} pages — text: {stats['text']}, ocr: {stats['ocr']}, docling: {stats['docling']}, empty: {stats['empty']}")
    print(f"  Text length: {len(text):,} characters")

    if not text.strip():
        print("  Skipping (no text extracted).")
        continue

    # Step 2: Feed extracted text to the KG pipeline
    result = await kg_builder_pdf.run_async(
        text=text,
        document_metadata={"source": pdf_path.name, "paper": pdf_path.stem},
    )
    print(f"  Done! {result.result}")

print("\nAll PDFs processed!")

## Step 4: Entity Resolution

Multiple PDFs will often mention the same entities with slightly different names. Entity resolution merges these duplicates. See Notebook 2 for a description of the resolver options.

In [ ]:
from neo4j_graphrag.experimental.components.resolver import (
    SinglePropertyExactMatchResolver,
)

resolver = SinglePropertyExactMatchResolver(driver, neo4j_database=NEO4J_DATABASE)
result = await resolver.run()
print(f"Resolved {result.number_of_nodes_to_resolve} entities → {result.number_of_created_nodes} merged nodes")

In [ ]:
INTERNAL_LABELS = ["__KGBuilder__", "__Entity__", "Chunk", "Document"]

# See the extracted entities with clean labels
records, _, _ = driver.execute_query(
    """
    MATCH (n:__Entity__)
    WITH [l IN labels(n) WHERE NOT l IN $internal][0] AS entity_type, n.name AS name
    RETURN entity_type, name
    ORDER BY entity_type, name
    """,
    {"internal": INTERNAL_LABELS},
    database_=NEO4J_DATABASE,
)

print("=== Extracted Entities ===")
current_type = None
for record in records:
    if record['entity_type'] != current_type:
        current_type = record['entity_type']
        print(f"\n  [{current_type}]")
    print(f"    - {record['name']}")

In [ ]:
# See extracted relationships with clean labels
records, _, _ = driver.execute_query(
    """
    MATCH (a:__Entity__)-[r]->(b:__Entity__)
    RETURN [l IN labels(a) WHERE NOT l IN $internal][0] AS from_type, a.name AS from_name,
           type(r) AS relationship,
           [l IN labels(b) WHERE NOT l IN $internal][0] AS to_type, b.name AS to_name
    ORDER BY relationship, from_name
    """,
    {"internal": INTERNAL_LABELS},
    database_=NEO4J_DATABASE,
)

print(f"=== Knowledge Graph Triples ({len(records)}) ===")
for record in records:
    print(
        f"  ({record['from_type']}: {record['from_name']}) "
        f"-[{record['relationship']}]-> "
        f"({record['to_type']}: {record['to_name']})"
    )

## Step 5: Graph Summary

In [ ]:
# Entity types across all papers
records, _, _ = driver.execute_query(
    """
    MATCH (n:__Entity__)
    WITH [l IN labels(n) WHERE NOT l IN $internal][0] AS entity_type, count(n) AS count
    RETURN entity_type, count
    ORDER BY count DESC
    """,
    {"internal": INTERNAL_LABELS},
    database_=NEO4J_DATABASE,
)

print("=== Entity Types Across All PDFs ===")
total = 0
for record in records:
    print(f"  {record['entity_type']}: {record['count']}")
    total += record['count']
print(f"\n  Total entities: {total}")

In [ ]:
# Documents in the graph
records, _, _ = driver.execute_query(
    """
    MATCH (d:Document)
    RETURN d.path AS document, d.source AS source
    """,
    database_=NEO4J_DATABASE,
)

print("=== Documents in the Graph ===")
for record in records:
    print(f"  {record['document']} (source: {record['source']})")

## Step 6: Query Across Papers

Since all papers are in the same graph, we can query across them!

In [ ]:
# Sample relationships across all documents
records, _, _ = driver.execute_query(
    """
    MATCH (a:__Entity__)-[r]->(b:__Entity__)
    RETURN [l IN labels(a) WHERE NOT l IN $internal][0] AS from_type, a.name AS from_name,
           type(r) AS rel, b.name AS to_name
    ORDER BY from_name
    LIMIT 25
    """,
    {"internal": INTERNAL_LABELS},
    database_=NEO4J_DATABASE,
)

print("=== Sample Relationships (first 25) ===")
for record in records:
    print(f"  ({record['from_type']}) {record['from_name']} -[{record['rel']}]-> {record['to_name']}")

## Summary

In this notebook you learned how to:
- Extract text from PDFs using PyMuPDF with automatic OCR fallback
- Feed extracted text into `SimpleKGPipeline` with `from_pdf=False`
- Define schemas for research papers (or let the LLM propose one)
- Build a combined graph from multiple papers

**Key takeaway:** By extracting text yourself with PyMuPDF, you get reliable extraction from both text-based and scanned PDFs — including automatic OCR for image-based pages.

**Next up:** [Notebook 4 — Building a Knowledge Graph from Web Pages](./04_kg_from_web.ipynb)

In [ ]:
driver.close()